In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report


In [ ]:
df = pd.read_csv('Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [ ]:
df.columns

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Amount Paid', 'Is Laundering',
       'Amount_Received_USD', 'Amount_Paid_USD',
       ...
       'Payment Currency_Yuan', 'Payment Format_ACH', 'Payment Format_Bitcoin',
       'Payment Format_Cash', 'Payment Format_Cheque',
       'Payment Format_Credit Card', 'Payment Format_Reinvestment',
       'Payment Format_Wire', 'Account_Same', 'Bank_Same'],
      dtype='object', length=109)

In [ ]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Columns: 104 entries, Amount Received to Bank_Same
dtypes: float64(101), int64(3)
memory usage: 3.9 GB


In [ ]:
random_state = 42

In [ ]:
fraud = df[df['Is Laundering'] == 1]
non_fraud = df[df['Is Laundering'] == 0]
len(fraud), len(non_fraud)

(5177, 5073168)

In [ ]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,US Dollar_Amount_Paid_Mean,US Dollar_Amount_Paid_Median,US Dollar_Amount_Received_Mean,US Dollar_Amount_Received_Median,Bitcoin_Amount_Paid_Mean,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
4563287,1439.79,1439.79,0,3.829841e+02,3.829841e+02,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4216770,33.09,33.09,0,3.309000e+01,3.309000e+01,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
2992877,263.34,263.34,0,3.807896e+01,3.807896e+01,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1499016,17154983.76,17154983.76,0,1.747064e+07,1.747064e+07,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
510282,8499.22,8499.22,0,2.501320e+03,2.501320e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3992741,5722.87,5722.87,0,5.722870e+03,5.722870e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
44465,82208.92,82208.92,0,8.220892e+04,8.220892e+04,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0
1173046,6837.09,6837.09,0,5.196872e+03,5.196872e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4125810,3261.27,3261.27,0,3.763506e+03,3.763506e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0


In [ ]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,US Dollar_Amount_Paid_Mean,US Dollar_Amount_Paid_Median,US Dollar_Amount_Received_Mean,US Dollar_Amount_Received_Median,Bitcoin_Amount_Paid_Mean,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,142.21,142.21,0,141.427845,141.427845,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
1,8.12,8.12,0,8.269408,8.269408,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
2,234.56,234.56,0,233.269920,233.269920,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
3,19640.00,19640.00,0,20001.376000,20001.376000,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4,8.38,8.38,0,8.380000,8.380000,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56942,78922.59,78922.59,0,3906.668205,3906.668205,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
56943,89535.13,89535.13,0,635.699423,635.699423,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
56944,207870.51,207870.51,0,2598.381375,2598.381375,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
56945,29.48,29.48,0,29.317860,29.317860,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0


In [ ]:
X = df_reduced.drop(columns='Is Laundering')
y = df_reduced['Is Laundering']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled

array([[-0.01368816, -0.01278547, -0.01008913, ..., -0.18157169,
        -0.34846905, -0.38402302],
       [-0.01368897, -0.01278628, -0.01009458, ..., -0.18157169,
        -0.34846905, -0.38402302],
       [-0.01368909, -0.0127864 , -0.01009542, ..., -0.18157169,
        -0.34846905, -0.38402302],
       ...,
       [-0.01368367, -0.01278097, -0.01005863, ..., -0.18157169,
        -0.34846905, -0.38402302],
       [-0.01253343, -0.01162778, -0.01003977, ..., -0.18157169,
        -0.34846905, -0.38402302],
       [-0.01368867, -0.01278597, -0.01009256, ..., -0.18157169,
        -0.34846905, -0.38402302]], shape=(45557, 103))

In [ ]:
X_test_scaled = scaler.transform(X_test)

In [ ]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

### RF with scaled data & recall as param tuning metric

In [ ]:
# Parameter tuning
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        # scoring="f1",
        scoring='recall',
        n_jobs=-1,
    )
    # f1 = scores.mean()
    recall = scores.mean()
    
    # return f1
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_df)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-14 14:20:48,392] A new study created in memory with name: no-name-65efc86f-277b-40e7-beb9-528fedfb91a3
[I 2026-04-14 14:20:54,968] Trial 0 finished with value: 0.7083527094514664 and parameters: {'n_estimators': 144, 'max_depth': 28, 'min_samples_split': 4}. Best is trial 0 with value: 0.7083527094514664.
[I 2026-04-14 14:21:00,310] Trial 1 finished with value: 0.7706456180836 and parameters: {'n_estimators': 148, 'max_depth': 25, 'min_samples_split': 7}. Best is trial 1 with value: 0.7706456180836.
[I 2026-04-14 14:21:02,198] Trial 2 finished with value: 0.8063766367720844 and parameters: {'n_estimators': 116, 'max_depth': 2, 'min_samples_split': 7}. Best is trial 2 with value: 0.8063766367720844.
[I 2026-04-14 14

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
88,88,0.852490,2026-04-14 14:24:11.079564,2026-04-14 14:24:13.947634,0 days 00:00:02.868070,9,5,124,COMPLETE
91,91,0.852249,2026-04-14 14:24:20.348595,2026-04-14 14:24:23.184013,0 days 00:00:02.835418,9,5,126,COMPLETE
96,96,0.852008,2026-04-14 14:24:34.171165,2026-04-14 14:24:36.727563,0 days 00:00:02.556398,9,5,103,COMPLETE
53,53,0.852007,2026-04-14 14:22:34.775041,2026-04-14 14:22:37.142433,0 days 00:00:02.367392,9,6,122,COMPLETE
93,93,0.851766,2026-04-14 14:24:25.953199,2026-04-14 14:24:28.674108,0 days 00:00:02.720909,9,5,118,COMPLETE


In [ ]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model.fit(X_train_scaled, y_train)

,n_estimators,124
,criterion,'gini'
,max_depth,9
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.93      0.96     10355
           1       0.55      0.85      0.67      1035

    accuracy                           0.92     11390
   macro avg       0.77      0.89      0.81     11390
weighted avg       0.94      0.92      0.93     11390



### RF With unscaled Data and recall as param tuning metric

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_unscaled_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_unscaled_df)

[I 2026-04-14 14:51:01,029] A new study created in memory with name: no-name-81346362-09b7-423e-a1de-c904b184b1e7
[I 2026-04-14 14:51:01,605] Trial 0 finished with value: 0.8162730255298568 and parameters: {'n_estimators': 75, 'max_depth': 2, 'min_samples_split': 6}. Best is trial 0 with value: 0.8162730255298568.
[I 2026-04-14 14:51:02,244] Trial 1 finished with value: 0.8042033064690012 and parameters: {'n_estimators': 61, 'max_depth': 3, 'min_samples_split': 9}. Best is trial 0 with value: 0.8162730255298568.
[I 2026-04-14 14:51:04,307] Trial 2 finished with value: 0.8232766909669411 and parameters: {'n_estimators': 71, 'max_depth': 17, 'min_samples_split': 10}. Best is trial 2 with value: 0.8232766909669411.
[I 2026-04-14 14:51:05,011] Trial 3 finished with value: 0.8046861068862432 and parameters: {'n_estimators': 49, 'max_depth': 5, 'min_samples_split': 5}. Best is trial 2 with value: 0.8232766909669411.
[I 2026-04-14 14:51:05,320] Trial 4 finished with value: 0.8399360733786704 

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
82,82,0.852007,2026-04-14 14:54:10.362024,2026-04-14 14:54:13.319674,0 days 00:00:02.957650,9,10,181,COMPLETE
93,93,0.851766,2026-04-14 14:54:39.923495,2026-04-14 14:54:42.850984,0 days 00:00:02.927489,9,8,175,COMPLETE
64,64,0.851766,2026-04-14 14:53:19.486939,2026-04-14 14:53:22.685829,0 days 00:00:03.198890,9,8,193,COMPLETE
71,71,0.851766,2026-04-14 14:53:38.013142,2026-04-14 14:53:40.987306,0 days 00:00:02.974164,9,8,175,COMPLETE
97,97,0.851766,2026-04-14 14:54:50.809066,2026-04-14 14:54:53.633662,0 days 00:00:02.824596,9,8,175,COMPLETE


In [ ]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_unscaled_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_unscaled_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_unscaled_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model.fit(X_train, y_train)

,n_estimators,181
,criterion,'gini'
,max_depth,9
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.93      0.96     10355
           1       0.55      0.84      0.67      1035

    accuracy                           0.92     11390
   macro avg       0.77      0.89      0.81     11390
weighted avg       0.94      0.92      0.93     11390



In [ ]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### No scaling, f1 as eval metric

In [ ]:
# Parameter tuning
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    )
    f1 = scores.mean()
    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 14:44:46,858] A new study created in memory with name: no-name-bd465fdb-5a7e-40a5-8419-cd9e25ede6c2
[I 2026-04-14 14:44:48,804] Trial 0 finished with value: 0.6689164323718346 and parameters: {'n_estimators': 35, 'max_depth': 5, 'min_samples_split': 10}. Best is trial 0 with value: 0.6689164323718346.
[I 2026-04-14 14:44:50,992] Trial 1 finished with value: 0.667854481318131 and parameters: {'n_estimators': 115, 'max_depth': 4, 'min_samples_split': 4}. Best is trial 0 with value: 0.6689164323718346.
[I 2026-04-14 14:44:54,599] Trial 2 finished with value: 0.6905742563524027 and parameters: {'n_estimators': 92, 'max_depth': 18, 'min_samples_split': 3}. Best is trial 2 with value: 0.6905742563524027.
[I 2026-04-14 14:44:56,789] Trial 3 finished with value: 0.6655179729953378 and parameters: {'n_estimators': 95, 'max_depth': 5, 'min_samples_split': 5}. Best is trial 2 with value: 0.6905742563524027.
[I 2026-04-14 14:44:59,206] Trial 4 finished with value: 0.6888239707782965 

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
25,25,0.691736,2026-04-14 14:45:45.313099,2026-04-14 14:45:49.990891,0 days 00:00:04.677792,19,2,164,COMPLETE
2,2,0.690574,2026-04-14 14:44:50.993420,2026-04-14 14:44:54.599128,0 days 00:00:03.605708,18,3,92,COMPLETE
74,74,0.690505,2026-04-14 14:48:09.782988,2026-04-14 14:48:13.562496,0 days 00:00:03.779508,19,3,128,COMPLETE
93,93,0.690391,2026-04-14 14:49:36.881259,2026-04-14 14:49:42.633462,0 days 00:00:05.752203,19,5,177,COMPLETE
23,23,0.690294,2026-04-14 14:45:40.002773,2026-04-14 14:45:43.654819,0 days 00:00:03.652046,19,3,129,COMPLETE


In [ ]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model.fit(X_train, y_train)

,n_estimators,164
,criterion,'gini'
,max_depth,19
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96     10355
           1       0.62      0.79      0.69      1035

    accuracy                           0.94     11390
   macro avg       0.80      0.87      0.83     11390
weighted avg       0.95      0.94      0.94     11390



In [ ]:
f1_model_report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)

### Compare Models

In [ ]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['RF optimized for Recall', 'RF optimized for F1'])

In [ ]:
combined_df

precision    recall  f1-score  \
RF optimized for Recall 0              0.983590  0.931917  0.957056   
                        1              0.553515  0.844444  0.668707   
                        accuracy       0.923968  0.923968  0.923968   
                        macro avg      0.768552  0.888181  0.812882   
                        weighted avg   0.944509  0.923968  0.930854   
RF optimized for F1     0              0.978542  0.951231  0.964693   
                        1              0.618580  0.791304  0.694362   
                        accuracy       0.936699  0.936699  0.936699   
                        macro avg      0.798561  0.871268  0.829528   
                        weighted avg   0.945832  0.936699  0.940128   

                                           support  
RF optimized for Recall 0             10355.000000  
                        1              1035.000000  
                        accuracy          0.923968  
                        macro avg     11390.000000  
                        weighted avg  11390.000000  
RF optimized for F1     0             10355.000000  
                        1              1035.000000  
                        accuracy          0.936699  
                        macro avg     11390.000000  
                        weighted avg  11390.000000

In [ ]:
model.feature_importances_

array([6.62856785e-02, 6.05995047e-02, 9.03900169e-02, 1.01445840e-01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
      

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
importances = model.feature_importances_
feature_names = df.drop(columns='Is Laundering').columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_df

,Feature,Gini Importance
94,Payment Format_ACH,0.384748
3,Amount_Paid_USD,0.101446
97,Payment Format_Cheque,0.101113
2,Amount_Received_USD,0.090390
98,Payment Format_Credit Card,0.066978
0,Amount Received,0.066286
1,Amount Paid,0.060600
101,Account_Same,0.037487
96,Payment Format_Cash,0.018723
99,Payment Format_Reinvestment,0.018419
